# 01 - Data Acquisition & Quality Assessment

**Financial Portfolio Tracker** | Notebook 1 of 5

---

## Objective

This notebook fetches historical price data for all assets in our diversified portfolio and the SPY benchmark, performs thorough data quality checks, and saves clean parquet files for downstream analysis.

**Portfolio Composition:**

| Ticker | Name | Weight | Asset Class |
|--------|------|--------|-------------|
| VOO | Vanguard S&P 500 ETF | 30% | US Equity |
| VXUS | Vanguard Total Intl Stock ETF | 20% | International Equity |
| VWO | Vanguard FTSE EM ETF | 10% | Emerging Markets |
| BND | Vanguard Total Bond Market ETF | 20% | Fixed Income |
| VNQ | Vanguard Real Estate ETF | 10% | Real Estate |
| GLD | SPDR Gold Shares | 10% | Commodities |
| **SPY** | **SPDR S&P 500 ETF (benchmark)** | -- | -- |

**Why this allocation?** The portfolio follows a modern multi-asset approach: 60% equities (diversified across US, international, and emerging markets), 20% fixed income for stability, and 20% alternatives (real estate + commodities) for inflation hedging and diversification.

## Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime

# Project paths
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for d in [RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data will be saved to: {PROCESSED_DIR}")

## Portfolio Configuration

We define the portfolio matching the backend configuration in `portfolio_backend/config.py`. This ensures consistency between the analytical notebooks and the live dashboard.

In [ ]:
# Portfolio definition -- mirrors backend/portfolio_backend/config.py
PORTFOLIO = {
    "VOO":  {"name": "Vanguard S&P 500 ETF",              "weight": 0.30, "category": "US Equity"},
    "VXUS": {"name": "Vanguard Total Intl Stock ETF",      "weight": 0.20, "category": "International Equity"},
    "VWO":  {"name": "Vanguard FTSE Emerging Markets ETF", "weight": 0.10, "category": "Emerging Markets"},
    "BND":  {"name": "Vanguard Total Bond Market ETF",     "weight": 0.20, "category": "Fixed Income"},
    "VNQ":  {"name": "Vanguard Real Estate ETF",           "weight": 0.10, "category": "Real Estate"},
    "GLD":  {"name": "SPDR Gold Shares",                   "weight": 0.10, "category": "Commodities"},
}

BENCHMARK = "SPY"
PERIOD = "5y"  # 5 years of daily data

TICKERS = list(PORTFOLIO.keys())
ALL_TICKERS = TICKERS + [BENCHMARK]

print(f"Portfolio tickers: {TICKERS}")
print(f"Benchmark: {BENCHMARK}")
print(f"Total weight: {sum(v['weight'] for v in PORTFOLIO.values()):.0%}")

## Data Fetch with yfinance

We use `yf.download()` with `auto_adjust=True` to get split- and dividend-adjusted closing prices. This is critical for total-return calculations -- using unadjusted prices would understate returns for dividend-paying ETFs like BND and VNQ.

In [ ]:
# Download all tickers in a single call for efficiency
raw = yf.download(ALL_TICKERS, period=PERIOD, auto_adjust=True, progress=True)

print(f"\nRaw data shape: {raw.shape}")
print(f"Date range: {raw.index.min().date()} to {raw.index.max().date()}")
print(f"Trading days: {len(raw)}")
raw.head()

In [ ]:
# Extract adjusted close prices
# yf.download with multiple tickers returns MultiIndex columns: (Price, Ticker)
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].copy()

prices = prices[ALL_TICKERS]  # Ensure consistent column order
print(f"Prices DataFrame shape: {prices.shape}")
prices.tail()

## Exploratory Data Profile

Before any analysis, we need to understand the shape and quality of the raw data. Key questions:
1. Are there missing values (trading halts, delistings)?
2. Are the price ranges reasonable (no obvious data errors)?
3. Do all tickers cover the same date range?

In [ ]:
# Data types and basic info
print("=" * 60)
print("DATA TYPES")
print("=" * 60)
print(prices.dtypes)
print(f"\nIndex type: {type(prices.index).__name__}")
print(f"Index dtype: {prices.index.dtype}")

In [ ]:
# Descriptive statistics
desc = prices.describe().round(2)
desc.loc["range"] = desc.loc["max"] - desc.loc["min"]
desc.loc["cv_%"] = (desc.loc["std"] / desc.loc["mean"] * 100).round(2)
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
desc

## Missing Data Analysis

Gaps in price data can arise from:
- **Market holidays** (expected -- these are not true gaps since no ticker trades)
- **ETF-specific halts** (rare for major Vanguard ETFs)
- **Data provider issues** (yfinance occasionally drops rows)

We check for NaN values and identify the exact dates where gaps occur.

In [ ]:
# Missing value report
missing = prices.isnull().sum()
missing_pct = (missing / len(prices) * 100).round(3)

gap_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct,
    "first_valid": prices.apply(lambda x: x.first_valid_index()),
    "last_valid": prices.apply(lambda x: x.last_valid_index()),
})

print("=" * 60)
print("MISSING DATA REPORT")
print("=" * 60)
gap_report

In [ ]:
# Identify specific gap dates (if any)
for ticker in ALL_TICKERS:
    nan_dates = prices[prices[ticker].isnull()].index
    if len(nan_dates) > 0:
        print(f"\n{ticker} - {len(nan_dates)} missing dates:")
        for d in nan_dates[:10]:  # Show first 10
            print(f"  {d.date()}")
        if len(nan_dates) > 10:
            print(f"  ... and {len(nan_dates) - 10} more")
    else:
        print(f"{ticker} - No missing data")

## Price History Visualization

Let's visualize the OHLCV data for VOO (our largest holding at 30%) to verify the data looks correct and to understand recent market behaviour.

In [ ]:
# Fetch OHLCV for VOO specifically for detailed visualization
voo_ohlcv = yf.download("VOO", period=PERIOD, auto_adjust=True, progress=False)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    subplot_titles=("VOO - Adjusted Close Price", "Daily Trading Volume")
)

# Candlestick chart
fig.add_trace(
    go.Candlestick(
        x=voo_ohlcv.index,
        open=voo_ohlcv["Open"].squeeze(),
        high=voo_ohlcv["High"].squeeze(),
        low=voo_ohlcv["Low"].squeeze(),
        close=voo_ohlcv["Close"].squeeze(),
        name="VOO",
        increasing_line_color="#26a69a",
        decreasing_line_color="#ef5350",
    ),
    row=1, col=1
)

# Volume bar chart
fig.add_trace(
    go.Bar(
        x=voo_ohlcv.index,
        y=voo_ohlcv["Volume"].squeeze(),
        name="Volume",
        marker_color="rgba(38, 166, 154, 0.5)",
    ),
    row=2, col=1
)

fig.update_layout(
    title="VOO (Vanguard S&P 500 ETF) - OHLCV Data",
    height=600,
    showlegend=False,
    xaxis_rangeslider_visible=False,
    template="plotly_white",
)
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)

fig.show()

## Normalized Price Comparison

To compare assets with different price levels, we normalize all prices to 100 at the start of the period. This reveals relative performance at a glance.

In [ ]:
# Normalize prices to 100 at the start
normalized = (prices / prices.iloc[0] * 100)

fig = px.line(
    normalized,
    title="Normalized Price Performance (Base = 100)",
    labels={"value": "Indexed Price", "variable": "Ticker", "Date": ""},
    template="plotly_white",
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.add_hline(y=100, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="Starting value")

fig.show()

## Stock Splits & Dividend Events

Even though we use auto-adjusted prices, it is good practice to check for corporate actions. Large splits or special dividends could cause jumps in unadjusted data. Let's verify that yfinance handled adjustments correctly.

In [ ]:
# Check for splits and dividends
events_summary = []

for ticker in ALL_TICKERS:
    tk = yf.Ticker(ticker)
    
    # Splits
    splits = tk.splits
    n_splits = len(splits[splits != 0]) if splits is not None and len(splits) > 0 else 0
    
    # Dividends
    divs = tk.dividends
    n_divs = len(divs) if divs is not None else 0
    avg_div = divs.mean() if n_divs > 0 else 0
    
    events_summary.append({
        "ticker": ticker,
        "name": PORTFOLIO.get(ticker, {}).get("name", "Benchmark"),
        "splits_count": n_splits,
        "dividends_count": n_divs,
        "avg_dividend_usd": round(avg_div, 4),
    })

events_df = pd.DataFrame(events_summary)
print("=" * 60)
print("CORPORATE ACTIONS SUMMARY")
print("=" * 60)
events_df

In [ ]:
# Check for suspicious daily price jumps (>15% move could indicate data error)
daily_returns = prices.pct_change().dropna()

extreme_threshold = 0.15  # 15%
extreme_moves = (daily_returns.abs() > extreme_threshold).sum()

print("=" * 60)
print(f"EXTREME DAILY MOVES (>{extreme_threshold:.0%})")
print("=" * 60)
for ticker in ALL_TICKERS:
    count = extreme_moves[ticker]
    if count > 0:
        dates = daily_returns[daily_returns[ticker].abs() > extreme_threshold].index
        moves = daily_returns.loc[dates, ticker]
        print(f"\n{ticker}: {count} extreme moves")
        for d, m in zip(dates, moves):
            print(f"  {d.date()}: {m:+.2%}")
    else:
        print(f"{ticker}: No extreme moves (data looks clean)")

## Data Cleaning & Gap Handling

For any remaining NaN values (e.g., from different listing dates), we forward-fill gaps. This is the standard approach for financial time series -- it assumes the last known price persists until a new observation arrives.

In [ ]:
# Forward-fill any remaining NaN gaps, then drop leading NaN rows
prices_clean = prices.ffill().dropna(how="any")

print(f"Before cleaning: {prices.shape[0]} rows, {prices.isnull().sum().sum()} NaNs")
print(f"After cleaning:  {prices_clean.shape[0]} rows, {prices_clean.isnull().sum().sum()} NaNs")
print(f"Date range: {prices_clean.index.min().date()} to {prices_clean.index.max().date()}")

## Save to Parquet

We save the cleaned price data as a Parquet file -- it is columnar, compressed, and preserves dtypes perfectly. This will be the input for all subsequent notebooks.

In [ ]:
# Save cleaned prices
output_path = PROCESSED_DIR / "prices_clean.parquet"
prices_clean.to_parquet(output_path)

# Verify the save
verify = pd.read_parquet(output_path)
assert verify.shape == prices_clean.shape, "Shape mismatch after save!"
assert verify.isnull().sum().sum() == 0, "NaN values found after save!"

file_size_kb = output_path.stat().st_size / 1024
print(f"Saved: {output_path}")
print(f"Size: {file_size_kb:.1f} KB")
print(f"Shape: {verify.shape}")
print(f"Columns: {list(verify.columns)}")
print(f"Date range: {verify.index.min().date()} to {verify.index.max().date()}")

## Summary & Data Quality Verdict

**Data Quality Assessment:**

1. **Completeness**: All 7 tickers (6 portfolio + 1 benchmark) have been fetched successfully with 5 years of daily data.
2. **Consistency**: Prices are auto-adjusted for splits and dividends, making them suitable for total return calculations.
3. **Validity**: No extreme price jumps that would indicate data errors. All values are positive and within expected ranges.
4. **Timeliness**: Data extends to the most recent trading day.

**Business Context**: This diversified portfolio spans 6 asset classes with different risk/return profiles. In the subsequent notebooks we will analyze how these assets interact, quantify portfolio risk, and determine whether the current allocation is optimal.

**Next step**: Notebook 02 - Portfolio Construction & Weighted Returns.

In [ ]:
# Final data summary table
summary = pd.DataFrame({
    "ticker": ALL_TICKERS,
    "category": [PORTFOLIO.get(t, {}).get("category", "Benchmark") for t in ALL_TICKERS],
    "weight": [PORTFOLIO.get(t, {}).get("weight", "--") for t in ALL_TICKERS],
    "first_price": prices_clean.iloc[0].values,
    "last_price": prices_clean.iloc[-1].values,
    "total_return": ((prices_clean.iloc[-1] / prices_clean.iloc[0] - 1) * 100).round(2).values,
    "trading_days": [prices_clean[t].notna().sum() for t in ALL_TICKERS],
})

summary["first_price"] = summary["first_price"].round(2)
summary["last_price"] = summary["last_price"].round(2)

print("=" * 70)
print("FINAL DATA SUMMARY")
print("=" * 70)
summary